# 21 — Click Prediction with Proper Cross-Validation

Rigorous evaluation of cursor approach features for click prediction, fixing three methodological issues in notebook 15:
1. **Participant leakage** — random KFold puts same participant in train and test
2. **Scaler leakage** — StandardScaler.fit_transform() on all data before CV
3. **Missing intermediate baseline** — Position+dwell not reported

Follows Huang, White & Buscher (CHI 2012) leave-one-subject-out (LOSO) protocol.

**Output:** Updated numbers for paper §3.5 and §4.3, plus threshold analysis for the tautology fix.

In [ ]:
import json, sys, warnings
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import LeaveOneGroupOut, GroupKFold, KFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve,
    precision_recall_curve, brier_score_loss
)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance

warnings.filterwarnings('ignore', category=UserWarning)

plt.rcParams.update({
    'figure.figsize': (10, 6), 'font.size': 12,
    'axes.spines.top': False, 'axes.spines.right': False
})

PLOT_DIR = Path('.')
DATA_PATH = Path('../AdSERP/data/cursor-approach-features.json')

In [ ]:
# Load feature data
with open(DATA_PATH) as f:
    raw = json.load(f)

# Extract participant IDs
for r in raw:
    r['participant'] = r['trial_id'].split('-')[0]

# Build arrays
all_keys = raw[0].keys()
n = len(raw)

# Feature column groups
FEATURES_M1 = ['position']
FEATURES_M2 = ['position', 'total_dwell_ms']
FEATURES_M3 = ['position', 'total_dwell_ms',
               'min_dist', 'mean_dist', 'final_dist', 'retreat_dist',
               'dwell_in_proximity_ms', 'mean_approach_velocity',
               'max_approach_velocity', 'direction_changes', 'frac_decreasing']
FEATURES_M4 = ['min_dist', 'mean_dist', 'final_dist', 'retreat_dist',
               'dwell_in_proximity_ms', 'mean_approach_velocity',
               'max_approach_velocity', 'direction_changes', 'frac_decreasing']

def build_X(features):
    X = np.array([[r[f] for f in features] for r in raw], dtype=np.float64)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    return X

X_m1 = build_X(FEATURES_M1)
X_m2 = build_X(FEATURES_M2)
X_m3 = build_X(FEATURES_M3)
X_m4 = build_X(FEATURES_M4)
y = np.array([int(r['was_clicked']) for r in raw])

le = LabelEncoder()
groups = le.fit_transform([r['participant'] for r in raw])

print(f"Records: {n:,}")
print(f"Participants: {len(le.classes_)}")
print(f"Click rate: {y.mean():.3f} ({y.sum():,} / {n:,})")
print(f"Records/participant: median {np.median(np.bincount(groups)):.0f}, "
      f"range {np.bincount(groups).min()}–{np.bincount(groups).max()}")

## 2. Evaluation Harness

`Pipeline(StandardScaler, LogisticRegression)` ensures the scaler fits only on training data within each fold. Three CV strategies as sensitivity analysis:
- **LOSO** (47-fold): gold standard, following Huang et al. (CHI 2012)
- **GroupKFold(5)**: grouped by participant, larger test folds
- **Random KFold(5)**: reproduces notebook 15 numbers (labeled as leaky baseline)

In [ ]:
def evaluate_model(X, y, groups, cv, model_name=''):
    """Run CV with Pipeline. Returns per-fold metrics and out-of-fold predictions."""
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))
    ])
    
    auc_scores, ap_scores = [], []
    y_pred_all = np.full(len(y), np.nan)
    fold_labels = np.full(len(y), -1, dtype=int)
    
    for fold_idx, (train_ix, test_ix) in enumerate(cv.split(X, y, groups)):
        pipe.fit(X[train_ix], y[train_ix])
        proba = pipe.predict_proba(X[test_ix])[:, 1]
        y_pred_all[test_ix] = proba
        fold_labels[test_ix] = fold_idx
        
        # AUC undefined for single-class folds (some LOSO participants may have only non-clicks)
        if len(np.unique(y[test_ix])) == 2:
            auc_scores.append(roc_auc_score(y[test_ix], proba))
            ap_scores.append(average_precision_score(y[test_ix], proba))
    
    return {
        'auc_scores': np.array(auc_scores),
        'ap_scores': np.array(ap_scores),
        'y_pred_proba': y_pred_all,
        'y_true': y.copy(),
        'fold_labels': fold_labels,
        'model_name': model_name,
        'n_valid_folds': len(auc_scores),
    }

print("Evaluation harness ready.")

## 3. Run All 12 Conditions (4 models × 3 CV strategies)

In [ ]:
cv_strategies = {
    'LOSO (47-fold)': LeaveOneGroupOut(),
    'GroupKFold (5)': GroupKFold(n_splits=5),
    'Random KFold (5) [leaky]': KFold(n_splits=5, shuffle=True, random_state=42),
}

model_tiers = [
    ('M1: Position', X_m1),
    ('M2: Position + Dwell', X_m2),
    ('M3: Full (Pos+Dwell+Approach)', X_m3),
    ('M4: Approach Only', X_m4),
]

results = {}
for cv_name, cv in cv_strategies.items():
    results[cv_name] = {}
    for model_name, X_feat in model_tiers:
        r = evaluate_model(X_feat, y, groups, cv, model_name)
        results[cv_name][model_name] = r
        auc_m = r['auc_scores'].mean()
        auc_s = r['auc_scores'].std()
        ap_m = r['ap_scores'].mean()
        print(f"  {cv_name:30s} | {model_name:35s} | AUC {auc_m:.3f}±{auc_s:.3f} | AP {ap_m:.3f} | folds={r['n_valid_folds']}")

print("\nDone — all 12 conditions evaluated.")

## 4. Results Table

In [ ]:
# Build summary table
print(f"{'Model':<35s} | {'LOSO AUC':>10s} | {'LOSO AP':>10s} | {'GrpKF AUC':>10s} | {'Rand AUC':>10s} | {'Δ (leak)':>8s}")
print("-" * 110)

for model_name, _ in model_tiers:
    loso = results['LOSO (47-fold)'][model_name]
    gkf = results['GroupKFold (5)'][model_name]
    rkf = results['Random KFold (5) [leaky]'][model_name]
    
    loso_auc = f"{loso['auc_scores'].mean():.3f}±{loso['auc_scores'].std():.3f}"
    loso_ap = f"{loso['ap_scores'].mean():.3f}±{loso['ap_scores'].std():.3f}"
    gkf_auc = f"{gkf['auc_scores'].mean():.3f}±{gkf['auc_scores'].std():.3f}"
    rkf_auc = f"{rkf['auc_scores'].mean():.3f}±{rkf['auc_scores'].std():.3f}"
    delta = rkf['auc_scores'].mean() - loso['auc_scores'].mean()
    delta_s = f"+{delta:.3f}" if delta > 0 else f"{delta:.3f}"
    
    print(f"{model_name:<35s} | {loso_auc:>10s} | {loso_ap:>10s} | {gkf_auc:>10s} | {rkf_auc:>10s} | {delta_s:>8s}")

print()
print("Δ (leak) = Random KFold AUC − LOSO AUC. Positive = leakage inflation.")

## 5. Per-Participant AUC Distribution (LOSO M3)

Each point is one participant held out entirely during training. Shows how well the model generalizes across individual cursor styles.

In [ ]:
loso_m3 = results['LOSO (47-fold)']['M3: Full (Pos+Dwell+Approach)']
aucs = loso_m3['auc_scores']

fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot(aucs, vert=True, widths=0.4, positions=[1],
           boxprops=dict(linewidth=1.5), medianprops=dict(color='#c0392b', linewidth=2))
ax.scatter(np.ones(len(aucs)) + np.random.uniform(-0.08, 0.08, len(aucs)),
           aucs, alpha=0.5, s=30, color='#2c3e50', zorder=3)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Chance')
ax.set_ylabel('AUC-ROC')
ax.set_title(f'Per-Participant AUC (LOSO, M3)\n'
             f'Median={np.median(aucs):.3f}, IQR=[{np.percentile(aucs, 25):.3f}, {np.percentile(aucs, 75):.3f}]')
ax.set_xticks([1])
ax.set_xticklabels(['M3: Full Model'])
ax.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / 'plot21_loso_auc_distribution.png', dpi=150)
plt.show()

print(f"Per-participant AUC (N={len(aucs)} participants with valid folds):")
print(f"  Min:    {aucs.min():.3f}")
print(f"  Q1:     {np.percentile(aucs, 25):.3f}")
print(f"  Median: {np.median(aucs):.3f}")
print(f"  Q3:     {np.percentile(aucs, 75):.3f}")
print(f"  Max:    {aucs.max():.3f}")

## 6. Model Comparison (Paper Figure 4)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

model_names = [m[0] for m in model_tiers]
x = np.arange(len(model_names))
width = 0.25

for i, (cv_name, color, hatch) in enumerate([
    ('LOSO (47-fold)', '#2c3e50', ''),
    ('GroupKFold (5)', '#2980b9', ''),
    ('Random KFold (5) [leaky]', '#bdc3c7', '//'),
]):
    means = [results[cv_name][m]['auc_scores'].mean() for m in model_names]
    stds = [results[cv_name][m]['auc_scores'].std() for m in model_names]
    bars = ax.bar(x + i*width, means, width, yerr=stds, label=cv_name,
                  color=color, hatch=hatch, edgecolor='white', capsize=3)

ax.set_ylabel('AUC-ROC')
ax.set_title('Click Prediction: Model Tiers × CV Strategy')
ax.set_xticks(x + width)
ax.set_xticklabels(['M1\nPosition', 'M2\nPos+Dwell', 'M3\nFull', 'M4\nApproach Only'],
                    fontsize=10)
ax.legend(fontsize=9)
ax.set_ylim(0.5, 1.0)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'plot21_model_comparison.png', dpi=150)
plt.show()

## 7. Feature Importance

Logistic regression coefficients from full-data refit (descriptive — for paper narrative). Standardized features so magnitudes are comparable.

In [ ]:
pipe_full = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))
])
pipe_full.fit(X_m3, y)
coefs = pipe_full.named_steps['clf'].coef_[0]

# Sort by absolute value
feat_names = FEATURES_M3
order = np.argsort(np.abs(coefs))
sorted_names = [feat_names[i] for i in order]
sorted_coefs = coefs[order]

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#27ae60' if c > 0 else '#c0392b' for c in sorted_coefs]
ax.barh(range(len(sorted_coefs)), sorted_coefs, color=colors)
ax.set_yticks(range(len(sorted_names)))
ax.set_yticklabels(sorted_names, fontsize=10)
ax.set_xlabel('Standardized Coefficient (+ → click, − → skip)')
ax.set_title('M3 Feature Coefficients (full-data refit)')
ax.axvline(0, color='gray', linewidth=0.5)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'plot21_feature_coefficients.png', dpi=150)
plt.show()

print("Coefficients (sorted by |value|):")
for name, coef in zip(reversed(sorted_names), reversed(sorted_coefs)):
    direction = "→ click" if coef > 0 else "→ skip"
    print(f"  {name:30s}: {coef:+.3f} {direction}")

## 8. Calibration Curve

Are the predicted probabilities meaningful? A well-calibrated model means p=0.3 corresponds to ~30% actual click rate. This matters for the threshold analysis.

In [ ]:
loso_m3 = results['LOSO (47-fold)']['M3: Full (Pos+Dwell+Approach)']
y_true = loso_m3['y_true']
y_pred = loso_m3['y_pred_proba']

# Remove any NaN predictions (from single-class folds)
valid = ~np.isnan(y_pred)
y_t, y_p = y_true[valid], y_pred[valid]

fraction_pos, mean_pred = calibration_curve(y_t, y_p, n_bins=10, strategy='uniform')
brier = brier_score_loss(y_t, y_p)

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
ax.plot(mean_pred, fraction_pos, 's-', color='#2c3e50', markersize=8, label='M3 LOSO')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of positives')
ax.set_title(f'Calibration Curve (Brier score = {brier:.4f})')
ax.legend()
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'plot21_calibration.png', dpi=150)
plt.show()

print(f"Brier score: {brier:.4f} (lower = better calibrated)")

## 9. Threshold Analysis (Tautology Fix)

The four-class taxonomy currently defines "approached-rejected" as `approached AND NOT clicked` — making 0% click rate tautological. Here we replace the post-hoc split with a learned threshold from the classifier's predicted probabilities.

At the optimal threshold, non-clicked approached results are reclassified:
- `p_click > threshold` → **deferred candidate** (model thinks plausible click)
- `p_click ≤ threshold` → **evaluated-rejected** (empirically defined hard negative)

In [ ]:
# ROC with operating points
fpr, tpr, thresholds_roc = roc_curve(y_t, y_p)

# Youden's J
j_scores = tpr - fpr
j_idx = np.argmax(j_scores)
j_threshold = thresholds_roc[j_idx]

# F1-optimal threshold
precisions_f1, recalls_f1, thresholds_f1 = precision_recall_curve(y_t, y_p)
f1_scores = 2 * precisions_f1[:-1] * recalls_f1[:-1] / (precisions_f1[:-1] + recalls_f1[:-1] + 1e-10)
f1_idx = np.argmax(f1_scores)
f1_threshold = thresholds_f1[f1_idx]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# ROC
overall_auc = roc_auc_score(y_t, y_p)
ax1.plot(fpr, tpr, color='#2c3e50', linewidth=2, label=f'AUC = {overall_auc:.3f}')
ax1.plot(fpr[j_idx], tpr[j_idx], 'o', color='#e74c3c', markersize=10,
         label=f"Youden's J (t={j_threshold:.3f})")
ax1.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curve with Operating Points')
ax1.legend()

# Precision-Recall
ax2.plot(recalls_f1, precisions_f1, color='#2c3e50', linewidth=2)
ax2.plot(recalls_f1[f1_idx], precisions_f1[f1_idx], 'o', color='#e74c3c', markersize=10,
         label=f'F1-optimal (t={f1_threshold:.3f}, F1={f1_scores[f1_idx]:.3f})')
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curve')
ax2.legend()

plt.tight_layout()
plt.savefig(PLOT_DIR / 'plot21_roc_operating_points.png', dpi=150)
plt.show()

print(f"Youden's J threshold: {j_threshold:.3f}")
print(f"  TPR={tpr[j_idx]:.3f}, FPR={fpr[j_idx]:.3f}")
print(f"F1-optimal threshold: {f1_threshold:.3f}")
print(f"  F1={f1_scores[f1_idx]:.3f}")

In [ ]:
# Apply learned threshold to reclassify non-clicked approached results
# "Approached" = min_dist < 100px (same definition as notebook 15)

threshold = j_threshold  # Use Youden's J as primary

approached = np.array([r['min_dist'] < 100 for r in raw])
clicked = y.astype(bool)

# Categories (classifier-derived, not tautological)
cat = np.full(n, '', dtype='U30')
cat[clicked] = 'Clicked'

# Non-clicked, approached: split by predicted probability
nc_approached = ~clicked & approached & valid  # valid = has predictions
cat[nc_approached & (y_p_full > threshold)] = 'Deferred candidate'
cat[nc_approached & (y_p_full <= threshold)] = 'Evaluated-rejected'

# Non-clicked, not approached
cat[~clicked & ~approached] = 'No signal'

# Handle any remaining (approached but no valid prediction)
cat[cat == ''] = 'No signal'

# We need y_p aligned to all records, not just valid
y_p_full = loso_m3['y_pred_proba'].copy()

# Redo with full predictions
cat = np.full(n, '', dtype='U30')
cat[clicked] = 'Clicked'
has_pred = ~np.isnan(y_p_full)
nc_appr_pred = ~clicked & approached & has_pred
cat[nc_appr_pred & (y_p_full > threshold)] = 'Deferred candidate'
cat[nc_appr_pred & (y_p_full <= threshold)] = 'Evaluated-rejected'
cat[cat == ''] = 'No signal'

# Count
from collections import Counter
counts = Counter(cat)
print("=== Classifier-Derived Taxonomy (Tautology Fix) ===")
print(f"Threshold: {threshold:.3f} (Youden's J)")
print()
for label in ['Clicked', 'Deferred candidate', 'Evaluated-rejected', 'No signal']:
    c = counts[label]
    pct = c / n * 100
    # Mean predicted click probability for this class
    mask = cat == label
    if has_pred[mask].any():
        mean_p = np.nanmean(y_p_full[mask])
        print(f"  {label:25s}: {c:6,} ({pct:5.1f}%)  mean p(click) = {mean_p:.3f}")
    else:
        print(f"  {label:25s}: {c:6,} ({pct:5.1f}%)")

print()
print("Compare to post-hoc counts from notebook 15:")
print("  Clicked:             1,981")
print("  Approached-rejected: 2,280 (was tautological 0%)")
print("  Peripherally seen:   4,548")
print("  Unseen:              6,588")

## 10. Paper-Ready Summary

In [ ]:
print("=" * 70)
print("PAPER §3.5 / §4.3 — UPDATED NUMBERS")
print("=" * 70)
print()
print("Cross-validation: LOSO (47-fold, following Huang et al. CHI 2012)")
print("Pipeline: StandardScaler → LogisticRegression(balanced)")
print()

for model_name, _ in model_tiers:
    r = results['LOSO (47-fold)'][model_name]
    print(f"  {model_name:35s}: AUC {r['auc_scores'].mean():.3f} ± {r['auc_scores'].std():.3f}"
          f"  |  AP {r['ap_scores'].mean():.3f} ± {r['ap_scores'].std():.3f}"
          f"  ({r['n_valid_folds']} valid folds)")

print()
print("Per-participant generalization (LOSO M3):")
aucs = results['LOSO (47-fold)']['M3: Full (Pos+Dwell+Approach)']['auc_scores']
print(f"  Median AUC: {np.median(aucs):.3f}")
print(f"  IQR: [{np.percentile(aucs, 25):.3f}, {np.percentile(aucs, 75):.3f}]")
print(f"  Range: [{aucs.min():.3f}, {aucs.max():.3f}]")
print()

# Leakage comparison
for model_name, _ in model_tiers:
    loso_auc = results['LOSO (47-fold)'][model_name]['auc_scores'].mean()
    rand_auc = results['Random KFold (5) [leaky]'][model_name]['auc_scores'].mean()
    delta = rand_auc - loso_auc
    print(f"  Leakage Δ {model_name:35s}: {delta:+.3f}")

print()
print(f"Threshold for taxonomy (Youden's J): {j_threshold:.3f}")
print()
print("=" * 70)